# 🔬 TrustOCT-KD: Retinal OCT Disease Classification & Knowledge Distillation
## Calibration-Aware Knowledge Distillation with Explainability Preservation

**Reference Architecture**: ResNet50 + Multi-Scale Feature Fusion (MSF) + CBAM Attention → MobileNetV3 Student

---

## SECTION 1: Environment & Hardware Verification

In [ ]:
!nvidia-smi

import torch
print(f"\n✅ PyTorch Version: {torch.__version__}")
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ Device: {torch.cuda.get_device_name(0)}")

In [ ]:
!pip install -q kagglehub thop seaborn scikit-learn matplotlib tqdm opencv-python pandas
print("\n✅ All required packages installed!")

## SECTION 2: Repository Clone & Project Setup

In [ ]:
import os

if not os.path.exists('TrustOCT-KD'):
    !git clone https://github.com/Gnanapravallika/TrustOCT-KD.git

%cd TrustOCT-KD
print("\n✅ Repository ready. Root files:")
!ls

## SECTION 3: Dataset Ingestion (Direct Colab VM Disk)

In [ ]:
# Upload kaggle.json to download dataset automatically
from google.colab import files
uploaded = files.upload()

import os
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("\n✅ Kaggle API key configured!")

In [ ]:
# Download Kermany OCT dataset directly to Colab disk
!kaggle datasets download -d paultimothymooney/kermany2018 -p data/ --unzip
print("\n✅ Kermany dataset ready on Colab disk!")

## SECTION 4: Dataset Exploration & Class Distribution

In [ ]:
import sys, os
sys.path.append('.')

from trustoct.dataset import print_class_distributions

# Find dataset root directory
DATA_DIR = 'data'
for root, dirs, f in os.walk(DATA_DIR):
    if 'train' in dirs and 'test' in dirs:
        DATA_DIR = root
        break

# Print Split Class Distribution breakdown
print_class_distributions(DATA_DIR)

## SECTION 5: Global Training Hyperparameters

In [ ]:
# Hyperparameters matching reference setup
epochs = 20
lr = 1e-4
batch_size = 32

print(f"⚙️ Configured Hyperparameters:")
print(f"   Epochs:     {epochs}")
print(f"   Learning Rate: {lr}")
print(f"   Batch Size: {batch_size}")

## SECTION 6: Model Training Execution

Trains each baseline model and proposed architectures separately under identical hyperparameters.

### 1. Train `resnet50` (Baseline)

In [ ]:
from trustoct.training.trainer import run_experiment

resnet50_ckpt, resnet50_hist = run_experiment('resnet50', DATA_DIR, epochs=epochs, lr=lr, batch_size=batch_size)

### 2. Train `msf_resnet50` (+ MultiScale)

In [ ]:
msf_ckpt, msf_hist = run_experiment('resnet50_msf', DATA_DIR, epochs=epochs, lr=lr, batch_size=batch_size)

### 3. Train `msf_cbam_resnet50` (+ MultiScale + CBAM - Teacher Model)

In [ ]:
teacher_ckpt, teacher_hist = run_experiment('resnet50_msf_cbam', DATA_DIR, epochs=epochs, lr=lr, batch_size=batch_size)

### 4. Train `student_mobilenetv3` (Student Baseline w/o KD)

In [ ]:
student_no_kd_ckpt, student_no_kd_hist = run_experiment('student_mobilenetv3', DATA_DIR, epochs=epochs, lr=1e-3, batch_size=batch_size)

### 5. Train `student_kd_mobilenetv3` (+ Calibration-Aware KD - Proposed)

In [ ]:
from trustoct.models import build_model, build_student
from trustoct.dataset.oct_dataset import get_dataloaders
from trustoct.training.distillation_trainer import DistillationTrainer
from configs.config import Config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, batch_size=batch_size, num_workers=2)

# Load pre-trained teacher
teacher_model = build_model('resnet50_msf_cbam', num_classes=4, pretrained=False)
ckpt = torch.load(teacher_ckpt, map_location=device)
teacher_model.load_state_dict(ckpt['model_state_dict'])

# Instantiate student
student_kd_model = build_student('mobilenetv3', num_classes=4, pretrained=True)

# Distillation
distiller = DistillationTrainer(
    teacher_model=teacher_model,
    student_model=student_kd_model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    lr=1e-3,
    num_epochs=epochs,
    temperature=4.0,
    alpha=0.3, beta=0.5, gamma=0.2,
    experiment_name='student_KD_TrustOCT'
)

student_kd_ckpt, kd_hist = distiller.fit()

## SECTION 7: Comparative Evaluation & Results Generation

In [ ]:
from trustoct.evaluation.comparison import run_full_comparison, generate_layercam_comparison

# Load student baseline
student_no_kd_model = build_student('mobilenetv3', num_classes=4, pretrained=False)
ckpt = torch.load(student_no_kd_ckpt, map_location=device)
student_no_kd_model.load_state_dict(ckpt['model_state_dict'])

# Load student KD
student_kd_model = build_student('mobilenetv3', num_classes=4, pretrained=False)
ckpt = torch.load(student_kd_ckpt, map_location=device)
student_kd_model.load_state_dict(ckpt['model_state_dict'])

# Run comprehensive evaluation across all models
df_comparison = run_full_comparison(
    teacher_model=teacher_model,
    student_model=student_kd_model,
    student_no_kd_model=student_no_kd_model,
    test_loader=test_loader,
    device=device
)

## SECTION 8: Explainability & AOPC Faithfulness Verification

In [ ]:
# Generate LayerCAM & AOPC comparative visualizations
aopc_df = generate_layercam_comparison(
    teacher_model=teacher_model,
    student_model=student_kd_model,
    test_loader=test_loader,
    device=device
)

## SECTION 9: Export Artifacts & Results

In [ ]:
# Download results ZIP
!zip -r TrustOCT_Results.zip outputs/results/ outputs/visualizations/

from google.colab import files
files.download('TrustOCT_Results.zip')
print("\n✅ All results and figures downloaded successfully!")